# 05 — PSA Grading Standards + How to Train Model 2

This notebook explains:

- PSA criteria categories (centering/corners/edges/surface)
- What we can measure from photos
- How to build a training set from graded slab images
- How to reduce label noise and data leakage

**Reality:** PSA grading is not a published algorithm. We train a model to *predict the historical grade label* on slabbed cards.


## PSA centering standards (source)
PSA publishes centering thresholds by grade on their grading standards page.

- PSA 10: ~60/40 or better (front) and 75/25 (back), with notes/leeway.
- Lower grades allow more off-centering.

In the code, we compute *border ratio features* that correlate with centering.


In [ ]:
# Your feature extractor aligns with these ideas.
# It computes lr_ratio and tb_ratio (closer to 1.0 is better).


## How to train the grader (Model 2)

### Dataset
Use eBay sold listings of PSA-graded cards:
- label = grade parsed from title, ideally verified by OCR of the slab label
- images = front and back if possible

### Key risks
1. **Leakage**: If the same listing photo appears in train and test, metrics inflate.
2. **Card identity bias**: Charizard photos may get higher grades on average.
3. **Slab artifacts**: Model can learn slab label cues. You must crop/warp the *card area only*.

### Fixes
- Split by listing_id (and ideally seller)
- Detect slab label area and mask it
- Warp the card to canonical rectangle before feeding model


### Model architecture
Two inputs:
- Image CNN embedding (learns scratches, whitening, print lines)
- Engineered features (centering/corner/edge/surface) for interpretability

This tends to work better with smaller datasets than CNN-only.


In [ ]:
!sed -n '1,220p' src/pokemon_valuator/utils/psa_feature_extractor.py

In [ ]:
!sed -n '1,220p' src/pokemon_valuator/models/psa_grader_model.py

In [ ]:
!sed -n '1,260p' scripts/train_grader.py

## What makes this notebook "portfolio-grade"

This notebook is intentionally **visual and diagnostic**:
- Shows what each PSA criterion means in image terms
- Visualizes the engineered features you compute
- Helps you debug failure cases

Even if the ML model isn't perfect yet, being able to explain and visualize the scoring is a big differentiator.

## Load one image and run the PSA feature extractor

This section runs your **interpretable** feature pipeline (centering/corners/edges/surface).

In [ ]:
from pathlib import Path

sample = Path('data/raw/sample_query.jpg')
if not sample.exists():
    print('Add a sample image at:', sample)
else:
    print('Using:', sample)

In [ ]:
from src.pokemon_valuator.utils.psa_feature_extractor import PSAFeatureExtractor

extractor = PSAFeatureExtractor()

if sample.exists():
    feats = extractor.extract(sample)
    feats

## Visual diagnostics

A strong next step is to generate overlays that show:
- detected card quadrilateral (warp)
- border measurements for centering
- edge strips used for whitening
- corner crops used for sharpness

The code below prints intermediate debug images **if the extractor was implemented with debug hooks**.
If you don't see visuals, that's your next upgrade: add `debug_dir` outputs to the extractor.

In [ ]:
# (Optional) If your extractor supports debug outputs, write them here.
# Example future API:
# feats = extractor.extract(sample, debug_dir='data/interim/psa_debug')

from pathlib import Path

debug_dir = Path('data/interim/psa_debug')
if debug_dir.exists():
    print('Debug images found:', len(list(debug_dir.glob('*'))))
    for p in sorted(debug_dir.glob('*'))[:10]:
        print(' -', p)

## Training recommendations (short)

To train the grader well:
- Crop/warp to **card only** (avoid slab-label leakage)
- Deduplicate (pHash) + group split by item_id
- Evaluate with **within-1-grade accuracy**
- Review failure cases visually

This notebook becomes your living "lab notebook" for those experiments.